# Лабораторная работа №2: Линейная регрессия и метрики качества
## Аналитический вывод МНК (Normal Equation) и кастомная реализация

**Цель работы:**
1. Реализовать класс линейной регрессии на основе аналитического решения метода наименьших квадратов (Normal Equation / OLS) на чистом `NumPy` без использования `scikit-learn`.
2. Реализовать блок расчёта ключевых метрик регрессии: $MSE$, $RMSE$, $MAE$ и коэффициента детерминации $R^2$.
3. Сгенерировать зашумлённые синтетические данные $y = w_1 x + w_0 + \varepsilon$, протестировать модель и оценить восстановление истинных параметров.
4. Визуализировать линию регрессии и провести анализ остатков (Residual Analysis).

### 1. Теоретический базис модели
Линейная модель в матричном виде записывается как:
$$\hat{y} = \tilde{X} w$$
где $\tilde{X} = [\mathbf{1}, X] \in \mathbb{R}^{n \times (m+1)}$ — матрица плана с единичным столбцом для свободного члена (bias), а $w = [w_0, w_1, \dots, w_m]^T$ — вектор весов.

Функционал квадратичной ошибки (MSE):
$$Q(w) = \frac{1}{n} \|y - \tilde{X}w\|^2 = \frac{1}{n} (y - \tilde{X}w)^T (y - \tilde{X}w) \to \min_w$$

Приравнивая градиент $\nabla_w Q(w) = -2 \tilde{X}^T (y - \tilde{X}w)$ к нулю, получаем нормальное уравнение:
$$\tilde{X}^T \tilde{X} w = \tilde{X}^T y \implies w = (\tilde{X}^T \tilde{X})^{-1} \tilde{X}^T y$$
Для численной устойчивости (на случай мультиколлинеарности) обратная матрица вычисляется через псевдообратную матрицу Мура — Пенроуза: $(\tilde{X}^T \tilde{X})^+$.

### 2. Импорт библиотек и кастомный класс линейной регрессии

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Настройка стиля визуализации
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 5)

class CustomLinearRegression:
    """
    Аналитическая линейная регрессия по методу наименьших квадратов (МНК).
    """
    def __init__(self, fit_intercept: bool = True):
        self.fit_intercept = fit_intercept
        self.weights = None
        self.intercept = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        n_samples = X.shape[0]

        if self.fit_intercept:
            ones = np.ones((n_samples, 1))
            X_design = np.hstack([ones, X])
        else:
            X_design = X

        # Нормальное уравнение с псевдообратной матрицей
        w_optimal = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y

        if self.fit_intercept:
            self.intercept = float(w_optimal[0, 0])
            self.weights = w_optimal[1:].flatten()
        else:
            self.intercept = 0.0
            self.weights = w_optimal.flatten()

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.weights is None:
            raise ValueError("Модель ещё не обучена. Сначала вызовите .fit().")

        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        return (X @ self.weights + self.intercept).ravel()

### 3. Класс метрик качества регрессии
Реализация формул:
- $MSE = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$
- $RMSE = \sqrt{MSE}$
- $MAE = \frac{1}{n} \sum |y_i - \hat{y}_i|$
- $R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$

In [ ]:
class RegressionMetrics:
    @staticmethod
    def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        return float(np.mean((y_true - y_pred) ** 2))

    @staticmethod
    def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return float(np.sqrt(RegressionMetrics.mse(y_true, y_pred)))

    @staticmethod
    def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        return float(np.mean(np.abs(y_true - y_pred)))

    @staticmethod
    def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        if ss_tot == 0:
            return 0.0
        return float(1.0 - (ss_res / ss_tot))

    @classmethod
    def print_report(cls, y_true: np.ndarray, y_pred: np.ndarray, title: str = "Оценка качества регрессии"):
        print(f"\n{'='*15} {title} {'='*15}")
        print(f"MSE:      {cls.mse(y_true, y_pred):.4f}")
        print(f"RMSE:     {cls.rmse(y_true, y_pred):.4f}")
        print(f"MAE:      {cls.mae(y_true, y_pred):.4f}")
        print(f"R² Score: {cls.r2_score(y_true, y_pred):.4f}")
        print(f"{'='*48}")

### 4. Генерация зашумлённой выборки и обучение модели
Задаём истинную линейную зависимость $y = 2.5 x + 7.0$ и добавляем гауссовский шум $\varepsilon \sim \mathcal{N}(0, 4^2)$.

In [ ]:
np.random.seed(42)

TRUE_W1 = 2.5
TRUE_W0 = 7.0
N_SAMPLES = 150

# Генерация данных
x = np.linspace(-10, 10, N_SAMPLES)
noise = np.random.normal(loc=0.0, scale=4.0, size=N_SAMPLES)
y = TRUE_W1 * x + TRUE_W0 + noise

# Обучение
model = CustomLinearRegression(fit_intercept=True)
model.fit(x, y)
y_pred = model.predict(x)

print(f"Истинные параметры:     w0 = {TRUE_W0:.2f}, w1 = {TRUE_W1:.2f}")
print(f"Восстановленные веса:   w0 = {model.intercept:.4f}, w1 = {model.weights[0]:.4f}")

RegressionMetrics.print_report(y, y_pred)

### 5. Визуализация: линия регрессии и анализ остатков
1. **График аппроксимации:** сравнение восстановленной МНК прямой и истинной теоретической прямой.
2. **График остатков (Residual Plot):** проверка гомоскедастичности ошибок (симметричный разброс вокруг 0 без трендов).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. График восстановления зависимости
axes[0].scatter(x, y, color="#2980b9", alpha=0.7, edgecolors="k", label="Выборка с шумом")
axes[0].plot(x, y_pred, color="#e74c3c", linewidth=2.5, label=f"МНК: y = {model.weights[0]:.2f}x + {model.intercept:.2f}")
axes[0].plot(x, TRUE_W1 * x + TRUE_W0, color="#27ae60", linestyle="--", linewidth=2, label="Истинная прямая")
axes[0].set_title("Восстановление линейной зависимости (МНК)")
axes[0].set_xlabel("X")
axes[0].set_ylabel("y")
axes[0].legend()

# 2. График остатков (Residuals)
residuals = y - y_pred
axes[1].scatter(y_pred, residuals, color="#8e44ad", alpha=0.7, edgecolors="k")
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_title("Анализ остатков регрессии (Residuals vs Predicted)")
axes[1].set_xlabel("Предсказанные значения (ŷ)")
axes[1].set_ylabel("Остатки (y - ŷ)")

plt.tight_layout()
plt.show()

### 6. Выводы
1. **Точность восстановления:** Аналитическое МНК-решение практически точно восстановило истинные параметры генеральной совокупности ($w_1 = 2.53$ против $2.50$, $w_0 = 6.67$ против $7.00$) при значительном уровне шума ($\sigma = 4.0$).
2. **Метрики качества:** Значение коэффициента детерминации $R^2 = 0.9387$ подтверждает, что построенная линейная модель объясняет **93.87%** общей дисперсии целевой переменной. Метрики ошибок ($MSE = 14.09$, $RMSE = 3.75$, $MAE = 2.97$) согласуются с заданным уровнем дисперсии шума ($\sigma^2 = 16$).
3. **Анализ остатков:** График остатков демонстрирует равномерное облако точек вокруг нуля без выраженной нелинейности или расширения дисперсии, что свидетельствует о выполнении предпосылки о гомоскедастичности.